# 1. Dependencies Installation

In [ ]:
%%capture
%pip install unsloth

In [ ]:
%pip install datasets --q

# 2. Configurations

In [3]:
configurations = {
    "model_name": "/kaggle/input/models/akuseru59/llama3-2/other/default/1",
    "dtype": None,
    "max_seq_length": 512,
    "load_in_4bit": True,
    "max_new_tokens": 2,
    "map_path": "/kaggle/input/datasets/akuseru59/banking-intent-test/map.json",
    "test_path": "/kaggle/input/datasets/akuseru59/banking-intent-test/test.csv"
}

# 3. Data Preparation

In [4]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import json

test_path = configurations.get("test_path")
map_path = configurations.get("map_path")
model_path = configurations.get("model_name")

def get_category(path):
    with open(path, "r", encoding="utf-8") as inf:
        category_map = json.load(inf)

    return dict(category_map)

category_map = get_category(map_path)

def get_dataset(path):
    try:
        dataset = load_dataset("csv", data_files=path)["train"]

        def add_label(example):
            key = example["category"].strip().lower()
            example["label"] = category_map[key]
            return example

        dataset = dataset.map(add_label)

        return dataset

    except FileNotFoundError:
        raise ValueError(f"No file found at {path}")

test_set = get_dataset(test_path)

print(test_set.column_names)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

['text', 'category', 'label']


# 4. Inference Class

In [5]:
class IntentClassification():
    def __init__(self, model_path, category_map):
        self.model_path = model_path
        self.reverse_category_map = {v: k for k, v in category_map.items()}

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            self.model_path,
            max_seq_length=configurations.get("max_seq_length"),
            dtype=configurations.get("dtype"),
            load_in_4bit=configurations.get("load_in_4bit")
        )

        FastLanguageModel.for_inference(self.model)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.prompt = """
        ### Instruction:
        Classify the intent of the following banking request.

        ### Input:
        {}

        ### Response:
        Answer:"""

    def __call__(self, message):
        prompt = self.prompt.format(
            message
        )

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt"
        ).to(self.model.device)

        input_length = inputs.input_ids.shape[1]

        output = self.model.generate(
            **inputs,
            max_new_tokens=configurations["max_new_tokens"],
            do_sample=False,
            use_cache=True,
            pad_token_id=self.tokenizer.eos_token_id
        )

        prediction_text = self.tokenizer.decode(
            output[0][input_length:],
            skip_special_tokens=True
        ).strip()

        import re
        match = re.search(r"\d+", prediction_text)

        if match:
            pred_label = int(match.group())
        else:
            return "unknown"

        return self.reverse_category_map.get(pred_label, "unknown")

    def _evaluate(self, test_set):
        from tqdm.notebook import trange

        y_true = []
        y_pred = []

        total_samples = len(test_set)

        for i in trange(total_samples, desc="Evaluating"):
            example = test_set[i]
            text = example["text"]
            label = int(example["label"])

            prompt = """### Instruction:
            Classify the intent of the following banking request.
            
            ### Input:
            {}
            
            ### Response:
            Answer: <|label|>"""

            prompt = prompt.format(
                text
            )

            inputs = self.tokenizer(prompt, return_tensors="pt").to('cuda')

            input_length = inputs.input_ids.shape[1]

            output = self.model.generate(
                **inputs,
                max_new_tokens=configurations.get("max_new_tokens"),
                use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

            predicted_text = self.tokenizer.decode(
                output[0][input_length:],
                skip_special_tokens=True
            ).strip()

            numeric_id = ''.join(filter(str.isdigit, predicted_text))

            if numeric_id == "":
                pred_label = -1
            else:
                pred_label = int(numeric_id)

            y_true.append(label)
            y_pred.append(pred_label)
        
        from sklearn.metrics import accuracy_score, f1_score, classification_report

        accuracy = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        
        print(f"\nFinal Test Set Results ({total_samples} samples):")
        print(f"Accuracy:  {accuracy * 100:.2f}%")
        print(f"Macro F1:  {macro_f1 * 100:.2f}%")

        print("\nDetailed Classification Report:")
        print(classification_report(y_true, y_pred, zero_division=0))

# 5. Evaluate and Infer

In [6]:
classifier = IntentClassification(model_path, category_map)

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## 5.1. Evaluate

In [7]:
# classifier._evaluate(test_set)

Evaluating:   0%|          | 0/3080 [00:00<?, ?it/s]

Both `max_new_tokens` (=2) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/


Final Test Set Results (3080 samples):
Accuracy:  91.56%
Macro F1:  91.56%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        40
           1       1.00      0.97      0.99        40
           2       0.95      1.00      0.98        40
           3       1.00      0.97      0.99        40
           4       1.00      0.88      0.93        40
           5       0.88      0.70      0.78        40
           6       0.95      0.93      0.94        40
           7       0.92      0.82      0.87        40
           8       1.00      0.95      0.97        40
           9       0.97      0.95      0.96        40
          10       0.95      0.93      0.94        40
          11       0.88      0.88      0.88        40
          12       0.90      0.93      0.91        40
          13       0.95      0.97      0.96        40
          14       0.85      0.85      0.85        40
          15       0.88   

## 5.2. Infer

In [8]:
message = "Am I able to get a card in EU?"
predicted_label = classifier.__call__(message)


print(predicted_label)

Both `max_new_tokens` (=2) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


country_support
